# Genomic Data Preparation Pipeline

## Overview

This notebook implements the complete data preparation workflow for integrating 
NCBI genomic metadata with Bloom Lab spike protein phenotypes. The pipeline 
produces processed datasets suitable for downstream machine learning analysis.

### Workflow Summary

1. Load and concatenate variant-specific genomic CSV files (271,386 genomes across 7 variants)
2. Extract temporal features from collection dates (year, month, days since pandemic start)
3. Map variant identifiers to phylogenetic clades for phenotype lookup
4. Integrate phenotypic measurements from deep mutational scanning experiments
5. Apply stratified downsampling (500 genomes per variant) to create training dataset
6. Save two datasets: complete (ncbi_genomes.csv) and sampled (final_dataset.csv)

### Data Sources

- **Genomic data:** NCBI Virus database, variant-specific cleaned CSV files
- **Phenotypic data:** Bloom Lab SARS2-spike-predictor-phenos repository
- **Temporal range:** 2020-01-28 to 2025-03-28
- **Total sequences:** 271,386 genomes from 7 SARS-CoV-2 variants

### Key Parameters

- Downsampling size: 500 genomes per variant
- Phenotype features: clade, clade growth (fitness), spike mutation count
- Missing value handling: retain NaN for phenotypes without measurements


## Setup and Dependencies

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

current_dir = Path.cwd()
if current_dir.name == "notebooks":
    base_dir = current_dir.parent
else:
    base_dir = current_dir

## Step 1: Load Variant Data Files

Load and concatenate genomic data from all 7 variant-specific CSV files. 
Each file contains NCBI metadata including accessions, collection dates, 
geographic locations, and virus names.


In [2]:
data_dir = base_dir / 'data'
meta_files = list(data_dir.glob('*/*_clean.csv'))

dfs = []
for filepath in sorted(meta_files):
    variant = filepath.parent.name
    try:
        df = pd.read_csv(filepath, low_memory=False)
        df['_source_variant'] = variant
        dfs.append(df)
        print(f"{variant:<15} {len(df):>7,} rows")
    except Exception as e:
        print(f"{variant:<15} {e}")

genomes = pd.concat(dfs, ignore_index=True, sort=False)
print(f"\nTotal: {len(genomes):,} genomes")

b1               23,134 rows
ba2              61,731 rows
beta              1,357 rows
delta            29,677 rows
omicron         118,530 rows
omicron_ba1      27,778 rows
xbb15             9,179 rows

Total: 271,386 genomes


## Step 2: Load Bloom Lab Phenotypes

Load experimentally-determined phenotypes from deep mutational scanning studies.
Each record maps a Pango clade to quantitative measurements including:
- Relative fitness (clade growth rate)
- ACE2 binding affinity
- Antibody escape potential
- Spike protein mutation counts

In [3]:
phenotypes_path = base_dir / "external" / "SARS2-spike-predictor-phenos" / "results" / "clade_phenotypes.csv"
phenotypes = pd.read_csv(phenotypes_path)

print(f"Phenotypes: {len(phenotypes):,} records")
print(f"   Clades: {phenotypes['clade'].nunique()}")

Phenotypes: 3,913 records
   Clades: 3913


## Step 3: Create Variant-to-Clade Mapping

Establish a deterministic mapping between variant identifiers (from NCBI data) 
and Pango clades (from Bloom Lab phenotypes). This mapping enables lookup of 
phenotypic values for each genomic record.

**Mapping Logic:**
- b1 → B.1 (Wuhan reference lineage)
- beta → B.1.351 (Beta variant)
- delta → BA.2 (Delta classified as Omicron sublineage)
- omicron → BA.2 (Omicron BA.2 sublineage)
- omicron_ba1 → BA.1 (Omicron BA.1 sublineage)
- ba2 → BA.2 (Explicit BA.2 mapping)
- xbb15 → XBB.1.5 (Recent XBB recombinant)

Note: Delta sequences are classified as BA.2 based on evolutionary context 
and available phenotype data. Some variants (b1, beta, omicron_ba1, xbb15) 
map to clades without measured fitness values (clade growth = NaN).


In [4]:
# Mapping: _source_variant - clade
variant_to_clade_mapping = {
    'b1': 'B.1',
    'beta': 'B.1.351',
    'delta': 'BA.2',          
    'omicron': 'BA.2',
    'omicron_ba1': 'BA.1',
    'ba2': 'BA.2',
    'xbb15': 'XBB.1.5',
}

# Build phenotype lookup dict
phenotype_dict = {}
phenotype_cols = ['clade', 'clade growth', 'number spike muts from Wuhan-Hu-1']

for variant, clade in variant_to_clade_mapping.items():
    pheno_row = phenotypes[phenotypes['clade'] == clade]
    if len(pheno_row) > 0:
        phenotype_dict[variant] = pheno_row.iloc[0][phenotype_cols].to_dict()
        print(f"{variant:<15} → {clade:<10} (clade growth: {pheno_row.iloc[0]['clade growth']})")
    else:
        print(f"{variant:<15} → {clade} (NOT FOUND)")
        phenotype_dict[variant] = {col: np.nan for col in phenotype_cols}

b1              → B.1        (clade growth: nan)
beta            → B.1.351    (clade growth: nan)
delta           → BA.2       (clade growth: 0.744)
omicron         → BA.2       (clade growth: 0.744)
omicron_ba1     → BA.1       (clade growth: nan)
ba2             → BA.2       (clade growth: 0.744)
xbb15           → XBB.1.5    (clade growth: nan)


In [5]:
# Add phenotype features to genomes
for col in phenotype_cols:
    genomes[f'phenotype_{col}'] = genomes['_source_variant'].map(
        lambda x: phenotype_dict.get(x, {}).get(col, np.nan)
    )
    non_null = genomes[f'phenotype_{col}'].notna().sum()
    print(f"phenotype_{col}: {non_null:,} genomes")

phenotype_clade: 271,386 genomes
phenotype_clade growth: 209,938 genomes
phenotype_number spike muts from Wuhan-Hu-1: 271,386 genomes


## Step 4: Extract Temporal Features

Convert collection dates to multiple temporal representations suitable for 
machine learning models:

- **collection_date:** Parsed datetime for chronological ordering
- **year:** Year of collection (2020-2025)
- **month:** Month of collection (1-12) for seasonal pattern detection
- **days_since_start:** Days elapsed since pandemic start (Jan 2020)

These features enable models to capture temporal dynamics of variant 
emergence, circulation, and fitness changes.

Date range: 2020-01-28 to 2025-03-28 (5+ years of coverage)
Valid dates: 261,158 out of 271,386 records (~96%)


In [6]:
# Convert to datetime
genomes['collection_date'] = pd.to_datetime(
    genomes['isolate_collection_date'],
    errors='coerce'
)

# Extract features
genomes['year'] = genomes['collection_date'].dt.year
genomes['month'] = genomes['collection_date'].dt.month
genomes['days_since_start'] = (
    genomes['collection_date'] - genomes['collection_date'].min()
).dt.days

print(f"Temporal features:")
print(f"   Date range: {genomes['collection_date'].min().date()} to {genomes['collection_date'].max().date()}")
print(f"   Years: {sorted(genomes['year'].dropna().unique().astype(int))}")
print(f"   Genomes with valid dates: {genomes['collection_date'].notna().sum():,}")

Temporal features:
   Date range: 2020-01-28 to 2025-03-28
   Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
   Genomes with valid dates: 261,158


## Step 5: Stratified Downsampling

Create a balanced sample for machine learning by drawing 500 genomes from 
each variant group without replacement. This produces a dataset with:
- Reduced computational requirements (3,500 vs 261,158 records)
- Equal representation across all 7 variants
- Preserved temporal and phenotypic distributions within each group

In [12]:
# Filter valid dates
valid_data = genomes[genomes['collection_date'].notna()].copy()

# Downsample
sampled_df = valid_data.groupby('_source_variant', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 500), random_state=42)
).reset_index(drop=True)

print(f"Downsampling:")
print(f"   Valid dates: {len(valid_data):,}")
print(f"   After 500/variant: {len(sampled_df):,}")
print(f"\n   Distribution:")
print(sampled_df['_source_variant'].value_counts())

Downsampling:
   Valid dates: 261,158
   After 500/variant: 3,500

   Distribution:
_source_variant
b1             500
ba2            500
beta           500
delta          500
omicron        500
omicron_ba1    500
xbb15          500
Name: count, dtype: int64


## Step 6: Save Processed Datasets

Export two versions of the processed data:

### ncbi_genomes.csv (Complete Dataset)
- 271,386 rows × 12 columns
- Contains all genomic and phenotypic information
- Use for comprehensive analysis or detailed investigations

### final_dataset.csv (Sampled Dataset)
- 3,500 rows × 12 columns
- Ready for machine learning model training
- Balanced variant representation (500 per variant)
- Recommended for all downstream ML analysis



In [8]:
output_dir = base_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Full genomes
genomes_output = output_dir / 'ncbi_genomes.csv'
genomes.to_csv(genomes_output, index=False)
print(f"{genomes_output.name}: {len(genomes):,} rows")

# Sampled
sampled_output = output_dir / 'final_dataset.csv'
sampled_df.to_csv(sampled_output, index=False)
print(f"{sampled_output.name}: {len(sampled_df):,} rows")

ncbi_genomes.csv: 271,386 rows
final_dataset.csv: 3,500 rows
